# End-to-End Face Mask Classification Pipeline (Kompilt 4 Skenario)
Jalankan notebook ini di Kaggle. Notebook ini mencakup:
1. SVM Tanpa Augmentasi
2. SVM Dengan Augmentasi (On-the-fly)
3. MobileNetV2 Tanpa Augmentasi
4. MobileNetV2 Dengan Augmentasi


In [ ]:
import sys
import os
# Jika script diupload sebagai dataset, hilangkan tanda pagar di bawah ini dan sesuaikan nama foldernya
# sys.path.append('/kaggle/input/datasets/naufalakmalrizqulloh/pecede')

from config import *
from src.preprocess import download_data_from_kaggle, mobilenet_preprocessing, setup_generators
from src.feature_engineering import load_and_extract_features, scale_features, extract_canny, extract_dwt
from src.train import train_svm, train_mobilenet
from src.evaluate import evaluate_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np

# Buat direktori models agar tidak error
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Download Data
kaggle_dir = download_data_from_kaggle()
print("Data terhubung di:", kaggle_dir)

## Skenario 1: SVM Tanpa Augmentasi

In [ ]:
print("Loading & Extracting Train data (Unaugmented)...")
X_train_c, X_train_d, y_train = load_and_extract_features(kaggle_dir / "Train", require_preprocess=True)

print("\nLoading & Extracting Validation data...")
X_val_c, X_val_d, y_val = load_and_extract_features(kaggle_dir / "Validation", require_preprocess=True)

print("\nScaling Features...")
X_train_c_s, X_val_c_s, _, _ = scale_features(X_train_c, X_val_c, None)

print("\nTraining SVM (Canny - Unaugmented)...")
svm_model = train_svm(X_train_c_s, y_train, MODELS_DIR / 'svm_canny_unaug.pkl')

print("\nEvaluasi SVM (Canny - Unaugmented)...")
pred_svm = svm_model.predict(X_val_c_s)
evaluate_model(y_val, pred_svm, CLASSES, "SVM (Canny Unaug)")

## Skenario 2: SVM Dengan Augmentasi

In [ ]:
print("Menyiapkan Data Augmentasi untuk SVM...")
train_datagen_svm, _ = setup_generators(kaggle_dir)

train_gen_svm = train_datagen_svm.flow_from_directory(
    kaggle_dir / "Train", 
    target_size=IMG_SIZE_SVM, 
    batch_size=BATCH_SIZE, 
    class_mode='categorical',
    shuffle=True
)

X_train_aug_c, y_train_aug = [], []
total_batches = 313 # Sesuai dengan ~10000 gambar / 32 batch
print(f"Mengekstrak fitur dari {total_batches} batch hasil augmentasi...")

for i in range(total_batches):
    batch_x, batch_y = next(train_gen_svm)
    for j in range(len(batch_x)):
        img_uint8 = (batch_x[j] * 255.0).astype(np.uint8)
        img_gray = img_uint8.squeeze()
        X_train_aug_c.append(extract_canny(img_gray))
        y_train_aug.append(np.argmax(batch_y[j]))

X_train_aug_c = np.array(X_train_aug_c, dtype=np.float32)
y_train_aug = np.array(y_train_aug)

print("Scaling Fitur Augmentasi...")
X_train_aug_c_s, _, _, _ = scale_features(X_train_aug_c, None, None)

print("\nTraining SVM (Canny - Augmented)...")
svm_model_aug = train_svm(X_train_aug_c_s, y_train_aug, MODELS_DIR / 'svm_canny_aug.pkl')

print("\nEvaluasi SVM (Canny - Augmented)...")
pred_svm_aug = svm_model_aug.predict(X_val_c_s)
evaluate_model(y_val, pred_svm_aug, CLASSES, "SVM (Canny Augmented)")

## Setup Generator CNN

In [ ]:
datagen_train_aug = ImageDataGenerator(
    rotation_range=10, width_shift_range=0.2, height_shift_range=0.2,
    zoom_range=0.25, horizontal_flip=True, preprocessing_function=mobilenet_preprocessing
)
datagen_unaug = ImageDataGenerator(preprocessing_function=mobilenet_preprocessing)

val_gen_cnn = datagen_unaug.flow_from_directory(
    kaggle_dir / "Validation", target_size=IMG_SIZE_CNN, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)

## Skenario 3: MobileNetV2 Tanpa Augmentasi

In [ ]:
print("\n--- Training MobileNetV2 (TANPA Augmentasi) ---")
train_gen_unaug_cnn = datagen_unaug.flow_from_directory(
    kaggle_dir / "Train", target_size=IMG_SIZE_CNN, batch_size=BATCH_SIZE, class_mode='categorical'
)

mobilenet_model_unaug, _ = train_mobilenet(
    train_gen_unaug_cnn, val_gen_cnn, MODELS_DIR / 'mobilenet_unaug.h5', epochs=5
)

print("\nEvaluasi MobileNetV2 (TANPA Augmentasi)...")
pred_cnn_unaug_probs = mobilenet_model_unaug.predict(val_gen_cnn)
pred_cnn_unaug = np.argmax(pred_cnn_unaug_probs, axis=1)
evaluate_model(val_gen_cnn.classes, pred_cnn_unaug, CLASSES, "MobileNetV2 Unaugmented")

## Skenario 4: MobileNetV2 Dengan Augmentasi

In [ ]:
print("\n--- Training MobileNetV2 (DENGAN Augmentasi) ---")
train_gen_aug_cnn = datagen_train_aug.flow_from_directory(
    kaggle_dir / "Train", target_size=IMG_SIZE_CNN, batch_size=BATCH_SIZE, class_mode='categorical'
)

mobilenet_model_aug, _ = train_mobilenet(
    train_gen_aug_cnn, val_gen_cnn, MODELS_DIR / 'mobilenet_aug.h5', epochs=5
)

print("\nEvaluasi MobileNetV2 (DENGAN Augmentasi)...")
pred_cnn_aug_probs = mobilenet_model_aug.predict(val_gen_cnn)
pred_cnn_aug = np.argmax(pred_cnn_aug_probs, axis=1)
evaluate_model(val_gen_cnn.classes, pred_cnn_aug, CLASSES, "MobileNetV2 Augmented")